In [1]:
import warnings
warnings.filterwarnings('ignore')

from IPython.core.display import display, HTML
display(HTML('<style>.container {width: 90% !important; }</style>'))

import glob, pyBigWig, shutil, os
import pandas as pd

import sys
sys.path.append('crestgv/')
from crestgv import crestgv

import seaborn as sns
import matplotlib.pyplot as plt

## Functions to calculate "null" enrichment scores and p-values

In [7]:
def run_null_cgv(genetic, output, random_collection="", random_collection_path=""):
    # Run CREST-GV to get the enrichment scores
    cgv_null = crestgv(genetic=genetic, output=output, 
                       min_number_genetics=100,
                       collection_name=random_collection, in_house_collection_path=random_collection_path) 
    df_cgv_null = cgv_null.calculate_enrichment_score(lessNG=False, greater25k=False) 
    return 

def run_bootstrap(output, N_BOOTS, SEED): 
    # Read the intermediate scores to generate the null distribution
    rounds_null = sorted(glob.glob(output + "/rounds/statistics_intermediate_round*")) 
    dfs_null = []
    for r in rounds_null:
        df = pd.read_csv(r, sep='\t', index_col=0)
        df = df[['CREST-GV']]
        dfs_null.append(df)
        
    df_null = pd.concat(dfs_null)

    # Sample with replacement 10,000 times from df_null to generate a full null
    df_null_bootstrap = df_null.sample(n=N_BOOTS, replace=True, random_state=SEED)

    # Display the two plots of raw null and bootstrap null side by side
    # Add titles to each sub-plot
    fig, (ax1, ax2) = plt.subplots(1,2)
    sns.histplot(df_null['CREST-GV'], kde=False, ax=ax1)
    ax1.set_title("raw_null")
    sns.histplot(df_null_bootstrap['CREST-GV'], kde=True, ax=ax2)
    ax2.set_title("bootstrapped_null")
    fig.show()

    return df_null_bootstrap

def generate_pvalue(genetic, output, real_df_cgv, random_collection="", random_collection_path="", N_BOOTS=10000, SEED=42):
    # Get the null enrichment scores
    run_null_cgv(genetic = genetic, output = output, 
                 random_collection=random_collection, random_collection_path=random_collection_path)
    # Get the null bootstrap distribution (this will plot a figure as it runs)
    df_null_bootstrap = run_bootstrap(output = output, N_BOOTS=N_BOOTS, SEED=SEED)

    # for each real enrichment score count the number of times a value equal to or greater than the observed value was sampled in the bootstrap null and divide by the number of samples
    df_cgv_with_pvals = real_df_cgv
    df_cgv_with_pvals['empirical_pvalue'] = real_df_cgv['CREST-GV_all'].apply(lambda x: (df_null_bootstrap['CREST-GV'] >= x).sum()/N_BOOTS)

    return df_cgv_with_pvals

## Calculate enrichment scores for a range of genetics, super-PBMC

In [6]:
# Get the list of genetic files
gfiles = glob.glob("/project/Wellcome_Discovery/datashare/AVOCATO/genetics_lead/hg38/*.txt")
# Remove file folders and extensions
gfile_bases = [os.path.splitext(os.path.basename(file))[0].replace('_genetics_from_GWAScatalog') for file in gfiles]
gfile_bases

['08_gout',
 '01_type1diabetes',
 '10_CysticFibrosis',
 '13_IBD',
 '09_Waldenstrom_macroglobulinemia',
 '03_MS',
 '02_type2diabetes',
 '04_intelligence',
 '12_schizophrenia',
 '07_parkinsonDisease',
 '00_Astle',
 '11_bipolarDisorder',
 '06_leukocyteDisease',
 '05_bodyheight']

In [2]:
for genetic in gfile_bases:
    genetic = "/project/Wellcome_Discovery/datashare/AVOCATO/genetics_lead/hg38/" + gfile_bases + "_genetics_from_GWAScatalog.txt"
    
    # Get the un-normalised scores
    output = "normalise_scores_test/" + genetic + "_super_pbmc/"
    cgv = crestgv(genetic = genetic, 
                  output = output, min_number_genetics=100, collection_name = "super_pbmc")
    df_cgv = cgv.calculate_enrichment_score(lessNG=False, greater25k=False)
    
    # Get the "random" cell type scores for p-values and normalising
    random_collection_path = "/project/Wellcome_Discovery/svenkat/CREST-GV/data/perc_10"
    output = "normalise_scores_test/" + genetic + "_random_perc_10/"

    df_cgv_with_pvals = generate_pvalue(genetic = genetic, output = output, real_df_cgv = df_cgv,
                                        random_collection = "", random_collection_path = random_collection_path)
    df_cgv_with_pvals = df_cgv_with_pvals.sort_values('CREST-GV_all', ascending = False)
    

Removing NaN rows from loaded file ...
Total number of used variants in CREST-GV: 527
Calculating (1) total number of base-pairs within peaks and (2) total number of base-pairs within peaks divided by uniquely mappable base-pairs ...


100%|██████████| 19/19 [00:03<00:00,  5.18it/s]


Intersecting genetic with peak regions ...


100%|██████████| 19/19 [00:09<00:00,  1.94it/s]


For each fold, parallelised shuffling background ...
Calculating (1) total number of base-pairs within peaks and (2) total number of base-pairs within peaks divided by uniquely mappable base-pairs ...


...

Calculating statistics ...
Adding shuffled background ...


100%|██████████| 5/5 [00:16<00:00,  3.37s/it]


Calculating statistics ...
Adding shuffled background ...


100%|██████████| 5/5 [00:16<00:00,  3.26s/it]


Calculating statistics ...
Cleaning temporary files ...
Processing genetics finished.
